# OpenUSD Sandbox

Content from other [OpenUSD tutorials](https://openusd.org/release/tut_usd_tutorials.html), with some extensions.

In [ ]:
import numpy as np

from osm_scene.constants import GEOD_WGS84

minx = 13.3568753
miny = 52.5181898
maxx = 13.3763515
maxy = 52.5275873

# top-left > top-right > bottom-right
(x_az, y_az), _, distance_m = GEOD_WGS84.inv(
    [minx, minx],
    [maxy, maxy],
    [maxx, minx],
    [maxy, miny],
    return_back_azimuth=False,
)

x_m, y_m = np.ceil(distance_m).astype(int)

# could add some buffers

x_int = GEOD_WGS84.fwd_intermediate(minx, maxy, x_az, npts=x_m, del_s=1.0)
y_int = GEOD_WGS84.fwd_intermediate(minx, maxy, y_az, npts=y_m, del_s=1.0)

In [ ]:
mesh_lons, mesh_lats = np.meshgrid(x_int.lons, y_int.lats, copy=False)
mesh_ele = np.zeros(mesh_lons.shape)

faces_n_lats = mesh_ele.shape[0] - 1
faces_n_lons = mesh_ele.shape[1] - 1

In [ ]:
i = np.tile(
    np.tile([0, 1, 1, 0], (faces_n_lats, 1))
    + np.arange(0, faces_n_lats)[:, np.newaxis],
    (1, faces_n_lons),
)

In [ ]:
j = np.tile(
    (
        np.tile([0, 0, 1, 1], (faces_n_lons, 1))
        + np.arange(0, faces_n_lons)[:, np.newaxis]
    ).flatten(),
    (faces_n_lats, 1),
)

In [ ]:
import pyproj

LLA_TO_ECEF = pyproj.Transformer.from_crs(
    {"proj": "latlon", "ellps": "WGS84", "datum": "WGS84"},
    {"proj": "geocent", "ellps": "WGS84", "datum": "WGS84"},
)

x, y, z = LLA_TO_ECEF.transform(
    mesh_lons.flatten(),
    mesh_lats.flatten(),
    mesh_ele.flatten(),
)

In [ ]:
from pxr import Usd, UsdGeom

stage = Usd.Stage.CreateNew("sandbox.usda")
UsdGeom.Xform.Define(stage, "/Root")
mesh = UsdGeom.Mesh.Define(stage, "/Root/Ground")

# transform with quaternion / full transform so the result is flat on xy plane

mesh.GetPointsAttr().Set(
    list(map(list, zip(x - x.min(), y - y.min(), z - z.min(), strict=False))),
)
mesh.GetFaceVertexCountsAttr().Set(np.ones(faces_n_lats * faces_n_lons) * 4)
mesh.GetFaceVertexIndicesAttr().Set(
    np.ravel_multi_index((i.flatten(), j.flatten()), mesh_ele.shape),
)

stage.GetRootLayer().Save()

In [ ]:
del stage